# May

In [9]:
import re
import os
from pathlib import Path
import pandas as pd
import tqdm

# =========================================================
# HELPERS
# =========================================================
def make_unique_columns(columns):
    counts = {}
    new_cols = []

    for col in columns:
        col = str(col).strip()
        if col not in counts:
            counts[col] = 0
            new_cols.append(col)
        else:
            counts[col] += 1
            new_cols.append(f"{col}.{counts[col]}")
    return new_cols

# Define simple hardcoded overrides for known typos between File Initials -> Excel Initials
# These tell the script: "If you see 'VR' in a file, treat it as 'VRB' for blending with Excel."
INITIAL_OVERRIDES = {
    "DS": "DDS",  # Assuming DS is meant to map to Excel's DDS or DMS, using DDS randomly if no other clue, but we should do dynamic matching if possible.
    "VR": "VRB", 
    "AC": "AC" # just placeholder
    # TL might be totally untracked or a different typo
}

def get_initials(participant_id: str):
    if pd.isna(participant_id):
        return None
    return str(participant_id).strip().split("-")[-1]


def get_group(participant_id: str):
    if pd.isna(participant_id):
        return None
    return str(participant_id).strip().split("-")[0]


def expand_scan_field(scan_field):
    """
    Examples:
        '193-196,198' -> [193,194,195,196,198]
        '249, 251-254' -> [249,251,252,253,254]
        '173' -> [173]
        '2178.0' -> [2178]
        '' / NaN -> []
    """
    if scan_field is None:
        return []

    if isinstance(scan_field, pd.Series):
        vals = [v for v in scan_field.tolist() if pd.notna(v) and str(v).strip() != ""]
        if not vals:
            return []
        if len(vals) > 1:
            scan_field = vals[0]
        else:
            scan_field = vals[0]

    if pd.isna(scan_field):
        return []

    # Clean the string carefully
    s = str(scan_field)
    
    # Common OCR/Typo fixes: Replace '?' with empty, handle commas disguised as spaces
    s = s.replace("?", "")
    
    # Replace newlines or semicolons with commas to avoid merging numbers across lines
    s = s.replace("\n", ",").replace("\r", ",").replace(";", ",")
    # Remove spaces
    s = s.replace(" ", "")
    # Keep only digits, commas, hyphens, and dots
    s = re.sub(r'[^0-9,\-\.]', '', s)
    
    # Prevent strange edge cases like "29562959-2961" resulting from missing commas between ranges
    # Clean up double hyphens or hanging commas
    s = s.replace("--", "-").strip(",-")
    
    if not s:
        return []

    scans = []
    for part in s.split(","):
        if not part:
            continue
        try:
            if "-" in part:
                # If there are multiple dashes (e.g., date formats), ignore to prevent unpacking errors
                parts_split = part.split("-")
                if len(parts_split) == 2:
                    a_str, b_str = parts_split
                    if not a_str or not b_str:
                        continue
                        
                    # Handle decimals like 2178.0 by converting to float then int
                    a = int(float(a_str))
                    b = int(float(b_str))
                    
                    # Prevent insanely large ranges from hanging the notebook
                    if abs(a - b) > 500:
                        continue
                        
                    if a <= b:
                        scans.extend(range(a, b + 1))
                    else:
                        scans.extend(range(a, b - 1, -1))
            else:
                scans.append(int(float(part)))
        except (ValueError, TypeError):
            continue

    return scans


# =========================================================
# PATHS
# =========================================================
DATA_PATH = Path(r"D:\OCTA DICOM files may 2021")

EXCEL_FILE = Path.cwd() / "excel_filer" / "MAY.xlsx"
OUTPUT_FILE = Path.cwd() / "excel_filer" / "matched_filenames_full.xlsx"
UNMATCHED_FILE = Path.cwd() / "excel_filer" / "unmatched_filenames_full.xlsx"
MISSING_EXPECTED_FILE = Path.cwd() / "excel_filer" / "missing_expected_scans.xlsx"

# =========================================================
# READ EXCEL
# =========================================================
df = pd.read_excel(EXCEL_FILE)
df.columns = make_unique_columns(df.columns)
df["Date"] = pd.to_datetime(df["Date"], errors="coerce", dayfirst=True)

# =========================================================
# DEFINE COLUMN MAPPING
# =========================================================
phase_specs = [
    # Regular Hand
    {"column": "Hand Baseline", "phase": "Baseline", "protocol_area": "Hand", "condition": "Regular"},
    {"column": "Hand Isc",      "phase": "Isc",      "protocol_area": "Hand", "condition": "Regular"},
    {"column": "Hand PORH",     "phase": "PORH",     "protocol_area": "Hand", "condition": "Regular"},
    # Regular Foot
    {"column": "Foot Baseline", "phase": "Baseline", "protocol_area": "Foot", "condition": "Regular"},
    {"column": "Foot Isc",      "phase": "Isc",      "protocol_area": "Foot", "condition": "Regular"},
    {"column": "Foot PORH",     "phase": "PORH",     "protocol_area": "Foot", "condition": "Regular"},
    # Flavanol Hand
    {"column": "Hand Baseline Flavanol", "phase": "Baseline", "protocol_area": "Hand", "condition": "Flavanol"},
    {"column": "Hand Isc Flavanol",      "phase": "Isc",      "protocol_area": "Hand", "condition": "Flavanol"},
    {"column": "Hand PORH Flavanol",     "phase": "PORH",     "protocol_area": "Hand", "condition": "Flavanol"},
    # Flavanol Foot
    {"column": "Foot baseline Flavanol", "phase": "Baseline", "protocol_area": "Foot", "condition": "Flavanol"},
    {"column": "Foot Isc Flavanol",      "phase": "Isc",      "protocol_area": "Foot", "condition": "Flavanol"},
    {"column": "Foot PORH Flavanol",     "phase": "PORH",     "protocol_area": "Foot", "condition": "Flavanol"},
]

existing_phase_specs = [spec for spec in phase_specs if spec["column"] in df.columns]

# =========================================================
# BUILD LONG EXCEL TABLE
# =========================================================
excel_long_rows = []

pbar = tqdm.tqdm(df.iterrows(), total=len(df), desc="Processing Excel Rows")
for _, row in pbar:
    participant_id = row["Participant ID"]
    pbar.set_description(f"ID: {participant_id}")
    date_value = row["Date"]
    initials = get_initials(participant_id)
    group = get_group(participant_id)

    if pd.isna(date_value) or initials is None:
        continue
        
    date_val_normalized = pd.to_datetime(date_value).normalize()

    for spec in existing_phase_specs:
        scan_numbers = expand_scan_field(row[spec["column"]])

        for scan_number in scan_numbers:
            excel_long_rows.append({
                "participant_id": participant_id,
                "DP_or_HP": group,
                "initials": initials,
                "date": date_val_normalized,
                "scan_number": int(scan_number),
                "phase": spec["phase"],
                "protocol_area": spec["protocol_area"],
                "condition": spec["condition"],
                "excel_column": spec["column"],
            })

excel_long_df = pd.DataFrame(excel_long_rows)

# =========================================================
# PARSE FILENAMES
# =========================================================
# Made regex even more lenient to capture missing initials or different formatting styles
filename_pattern = re.compile(
    r'^(?P<initials>[A-Za-z\-]*?)_?'            # Now optional or might contain a dash
    r'(?P<side>Right|Left|right|left)_?'        # Handle optional underscores and lowercase
    r'(?P<bodypart>.*?)_'                       # Any bodypart
    r'(?P<Lnum>L\d+)_?'                         # Handle _L1047_ variants
    r'(S|s)(?P<scan_number>\d+)__'               # S or s
    r'(?P<date>\d{2}_\d{2}_\d{4}|\d{4}-\d{2}-\d{2})\.dcm$', # Acccept alternate date formats
    re.IGNORECASE
)

# If the above strict pattern continually fails, we can fall back to a simpler search anywhere in the filename
backup_pattern = re.compile(r'L\d+_[Ss](?P<scan_number>\d+)__(?P<date>\d{2}_\d{2}_\d{4})\.dcm')

file_rows = []
unmatched_patterns = []

for fname in tqdm.tqdm(os.listdir(DATA_PATH), desc="Parsing DICOM files"):
    if fname.startswith("."): continue
    
    m = filename_pattern.search(fname.strip()) 
    
    if not m:
        fallback_m = backup_pattern.search(fname.strip())
        if fallback_m:
            file_rows.append({
                "filename": fname,
                "initials": "UNKNOWN",  
                "side": "Unknown",
                "bodypart": "Unknown",
                "protocol_area": "Unknown",
                "Lnum": "Unknown",
                "scan_number": int(fallback_m.group("scan_number")),
                "date": pd.to_datetime(fallback_m.group("date"), format="%d_%m_%Y", errors="coerce").normalize(),
            })
        else:
            unmatched_patterns.append(fname)
        continue
        
    bodypart = m.group("bodypart") if m.group("bodypart") else ""
    bodypart = bodypart.strip()
    
    # Handle dates
    date_str = m.group("date")
    if "_" in date_str:
        file_date = pd.to_datetime(date_str, format="%d_%m_%Y", errors="coerce")
    else:
        file_date = pd.to_datetime(date_str, errors="coerce") 
    
    # Map protocol area
    protocol_area = "Hand" if "hand" in bodypart.lower() or "finger" in bodypart.lower() else "Foot"
    
    # Clean initials & Apply Typos Map
    initials = m.group("initials") if m.group("initials") else ""
    initials = initials.upper().replace("-", "")
    
    if initials == "DS":
        # Because we have both DDS and DMS missing, let's use the date to guess which one it is (or pick one)
        # Using a simplistic assumption: Let's let the `merge` dynamically find the closest match.
        # But for now, we'll map VR -> VRB explicitly
        pass
    
    if initials in INITIAL_OVERRIDES:
        initials = INITIAL_OVERRIDES[initials]

    file_rows.append({
        "filename": fname,
        "initials": initials,
        "side": m.group("side") if m.group("side") else "Unknown",
        "bodypart": bodypart,
        "protocol_area": protocol_area,
        "Lnum": m.group("Lnum") if m.group("Lnum") else "Unknown",
        "scan_number": int(m.group("scan_number")),
        "date": file_date.normalize() if pd.notna(file_date) else pd.NaT,
    })

files_df = pd.DataFrame(file_rows)

if len(unmatched_patterns) > 0:
    print(f"\n[WARNING] {len(unmatched_patterns)} files STILL completely failed the regex match! Here are some examples:")
    for un in unmatched_patterns[:10]:
        print(f"   -> {un}")

# =========================================================
# MERGE AND SAVE
# =========================================================
if not files_df.empty and not excel_long_df.empty:
    
    # To fix "date off by one" issues between filename dates and excel dates,
    # let's merge strictly on participant initials, scan number, and protocol area
    # If the initials are 'UNKNOWN' (from the fallback parser), we can try matching just by scan_number and area as a fallback.
    
    # standard merge
    matched_df = files_df.merge(excel_long_df, on=["initials", "scan_number", "protocol_area"], how="left", suffixes=('_file', '_excel'))
    
    # Identify files that still didn't match
    unmatched_mask = matched_df['participant_id'].isna()
    
    # Try a secondary fuzzy merge for files that failed to match (maybe initials were entirely mismatched like DS vs DDS vs DMS vs Unknown)
    # We will match those purely by `scan_number` and `protocol_area` against any unused excel rows
    if unmatched_mask.any():
        unmatched_subset = matched_df[unmatched_mask].drop(columns=['participant_id','DP_or_HP','phase','condition','date_excel','excel_column'], errors='ignore')
        
        # Used rows in excel:
        used_excel = matched_df[~unmatched_mask][['initials', 'scan_number', 'protocol_area']].drop_duplicates()
        
        # Unused rows in excel:
        # Instead of doing complex anti-joins, we just merge the unmatched on scan_number and protocol_area
        # and take the first hit. Scan numbers are generally globally unique across patients.
        fuzzy_matches = unmatched_subset.merge(
            excel_long_df, 
            on=["scan_number", "protocol_area"], 
            how="inner", 
            suffixes=('_file', '_excel')
        )
        
        # Only keep the first fuzzy match per file to avoid duplicates
        fuzzy_matches = fuzzy_matches.drop_duplicates(subset=['filename'])
        
        # Update the matched_df with these new fuzzy finds!
        for _, fuzzy_row in fuzzy_matches.iterrows():
            idx = matched_df[matched_df['filename'] == fuzzy_row['filename']].index
            if len(idx) > 0:
                # Plop in the found excel data
                matched_df.loc[idx, 'participant_id'] = fuzzy_row['participant_id']
                matched_df.loc[idx, 'DP_or_HP'] = fuzzy_row['DP_or_HP']
                matched_df.loc[idx, 'phase'] = fuzzy_row['phase']
                matched_df.loc[idx, 'condition'] = fuzzy_row['condition']
                matched_df.loc[idx, 'excel_column'] = fuzzy_row['excel_column']
                # override initials with what excel expected
                matched_df.loc[idx, 'initials'] = fuzzy_row['initials_excel'] if 'initials_excel' in fuzzy_row else fuzzy_row.get('initials_y', fuzzy_row.get('initials'))
        
    
    # Output columns list
    # Making sure to safely grab available date columns 
    out_cols = ["filename","participant_id","DP_or_HP","Lnum","phase","protocol_area","condition","bodypart","side", "scan_number","excel_column"]
    if 'date_file' in matched_df.columns: out_cols.append('date_file')
    if 'date_excel' in matched_df.columns: out_cols.append('date_excel')
    
    output_df = matched_df[out_cols].copy()
    output_df.to_excel(OUTPUT_FILE, index=False)

    unmatched_df = output_df[output_df["participant_id"].isna()].copy()
    unmatched_df.to_excel(UNMATCHED_FILE, index=False)

    # For missing expected, we want to know what's left in excel_long_df that isn't cleanly tracked to output_df
    # A simple way: find elements in excel that aren't in successful outputs.
    successful_excel = output_df[output_df['participant_id'].notna()]
    # Merge indicator
    merged_for_missing = excel_long_df.merge(
        successful_excel[['participant_id', 'scan_number', 'protocol_area']], 
        on=['participant_id', 'scan_number', 'protocol_area'], 
        how='left', indicator=True
    )
    
    missing_expected_df = merged_for_missing[merged_for_missing["_merge"] == "left_only"].copy()
    missing_expected_df.drop(columns=['_merge'], inplace=True)
    missing_expected_df.to_excel(MISSING_EXPECTED_FILE, index=False)

    print(f"\nSummary:")
    print(f"Total parsed files:   {len(files_df)}")
    print(f"Matched files:        {output_df['participant_id'].notna().sum()}")
    print(f"Unmatched files:      {len(unmatched_df)}")
    print(f"Missing (in Excel but no file): {len(missing_expected_df)}")
else:
    print("\nSummary: One or both of the dataframes are empty, skipping merge.")

Parsing DICOM files: 100%|██████████| 4736/4736 [00:00<00:00, 11918.34it/s]



Summary:
Total parsed files:   2620
Matched files:        2529
Unmatched files:      154
Missing (in Excel but no file): 1220


In [3]:
# Let's inspect WHY some files are still unmatched or missing

# unmatched_df comes from output_df, which does not have the 'initials' column 
# because we didn't include it in output_df's column slice. 
# We can just join back or display what's available:
print("--- Sample of 5 Unmatched DICOM Files (Found file, but no Excel match) ---")
display(unmatched_df[["filename", "scan_number", "protocol_area", "date_file"]].head(5))

print("\n--- Sample of 5 Missing Expected Scans (Found in Excel, but no DICOM file) ---")
display(missing_expected_df[["participant_id", "initials", "scan_number", "protocol_area", "excel_column"]].head(5))

# Check unique initials in both to see if we have typos (e.g., 'AKD' vs 'AK')
excel_inits = set(excel_long_df['initials'].dropna().unique())
file_inits = set(files_df['initials'].dropna().unique())

print("\nInitials in Excel but not in Files (Typo in Excel or missing entirely?):")
print(sorted(list(excel_inits - file_inits)))

print("\nInitials in Files but not in Excel (Typo in file name or unregistered patient?):")
print(sorted(list(file_inits - excel_inits)))

--- Sample of 5 Unmatched DICOM Files (Found file, but no Excel match) ---


,filename,scan_number,protocol_area,date_file
1,AC_Right Hallux_L1051_S2629__24_05_2021.dcm,2629,Foot,2021-05-24
7,AC_Right Hallux_L1051_S2626__24_05_2021.dcm,2626,Foot,2021-05-24
8,AC_Right Hallux_L1051_S2621__24_05_2021.dcm,2621,Foot,2021-05-24
13,AC_Right Hallux_L1051_S2620__24_05_2021.dcm,2620,Foot,2021-05-24
19,AC_Right Hallux_L1051_S2628__24_05_2021.dcm,2628,Foot,2021-05-24



--- Sample of 5 Missing Expected Scans (Found in Excel, but no DICOM file) ---


,participant_id,initials,scan_number,protocol_area,excel_column
24,HP-24-AKD,AKD,2185,Foot,Foot PORH
25,HP-24-AKD,AKD,2186,Foot,Foot PORH
26,HP-24-AKD,AKD,2187,Foot,Foot PORH
27,HP-24-AKD,AKD,2188,Foot,Foot PORH
28,HP-24-AKD,AKD,2189,Foot,Foot PORH



Initials in Excel but not in Files (Typo in Excel or missing entirely?):
['DMS']

Initials in Files but not in Excel (Typo in file name or unregistered patient?):
['TL']


In [5]:
# =========================================================
# ADD DEMOGRAPHICS DATA (AGE & SEX)
# =========================================================
demographics_data = [
    {"initials": "DG", "sex": "M", "age": 43, "group": "dp"},
    {"initials": "JR", "sex": "M", "age": 47, "group": "dp"},
    {"initials": "RH", "sex": "M", "age": 70, "group": "dp"},
    {"initials": "TA", "sex": "M", "age": 60, "group": "dp"},
    {"initials": "JF", "sex": "F", "age": 68, "group": "dp"},
    {"initials": "AC", "sex": "M", "age": 57, "group": "dp"},
    {"initials": "JP", "sex": "M", "age": 64, "group": "dp"},
    {"initials": "DC", "sex": "F", "age": 47, "group": "dp"},
    {"initials": "SR", "sex": "F", "age": 71, "group": "dp"},
    {"initials": "DH", "sex": "M", "age": 66, "group": "dp"},
    {"initials": "AT", "sex": "M", "age": 58, "group": "dp"},
    {"initials": "NL", "sex": "M", "age": 62, "group": "dp"},
    {"initials": "RM", "sex": "M", "age": 26, "group": "hp"},
    {"initials": "MB", "sex": "F", "age": 28, "group": "hp"},
    {"initials": "GU", "sex": "F", "age": 31, "group": "hp"},
    {"initials": "IT", "sex": "M", "age": 24, "group": "hp"},
    {"initials": "MF", "sex": "F", "age": 26, "group": "hp"},
    {"initials": "VM", "sex": "F", "age": 29, "group": "hp"},
    {"initials": "AF", "sex": "M", "age": 33, "group": "hp"},
    {"initials": "PM", "sex": "F", "age": 27, "group": "hp"},
    {"initials": "ES", "sex": "F", "age": 26, "group": "hp"},
    {"initials": "AG", "sex": "F", "age": 22, "group": "hp"},
    {"initials": "LM", "sex": "F", "age": 54, "group": "hp"},
    {"initials": "GW", "sex": "F", "age": 22, "group": "hp"},
    {"initials": "PC", "sex": "F", "age": 39, "group": "hp"},
    {"initials": "CH", "sex": "M", "age": 46, "group": "hp"},
    {"initials": "LT", "sex": "F", "age": 37, "group": "hp"},
    {"initials": "DMS", "sex": "F", "age": 38, "group": "hp"},
    {"initials": "DDS", "sex": "M", "age": 62, "group": "hp"},
    {"initials": "AKD", "sex": "M", "age": 26, "group": "hp"},
    {"initials": "NB", "sex": "F", "age": 56, "group": "hp"},
    {"initials": "BB", "sex": "M", "age": 68, "group": "hp"},
    {"initials": "JJ", "sex": "F", "age": 59, "group": "hp"},
    {"initials": "WL", "sex": "M", "age": 49, "group": "hp"},
    {"initials": "RL", "sex": "M", "age": 32, "group": "hp"},
    {"initials": "VRB", "sex": "F", "age": 54, "group": "hp"},
    {"initials": "VF", "sex": "F", "age": 44, "group": "hp"},
    {"initials": "GM", "sex": "M", "age": 62, "group": "hp"},
    {"initials": "DB", "sex": "M", "age": 56, "group": "hp"},
    {"initials": "LB", "sex": "F", "age": 55, "group": "hp"},
    {"initials": "HR", "sex": "M", "age": 33, "group": "hp"},
    {"initials": "HB", "sex": "F", "age": 56, "group": "hp"},
    {"initials": "HL", "sex": "F", "age": 56, "group": "hp"},
    {"initials": "DR", "sex": "F", "age": 49, "group": "hp"},
    {"initials": "PD", "sex": "M", "age": 41, "group": "hp"},
    {"initials": "AD", "sex": "F", "age": 43, "group": "hp"},
    {"initials": "JG", "sex": "F", "age": 61, "group": "hp"},
    {"initials": "MJ", "sex": "F", "age": 71, "group": "hp"},
    {"initials": "JK", "sex": "M", "age": 74, "group": "hp"},
    {"initials": "CSM", "sex": "F", "age": 67, "group": "hp"},
    {"initials": "TC", "sex": "M", "age": 61, "group": "hp"},
    {"initials": "CS", "sex": "F", "age": 60, "group": "hp"},
    {"initials": "MO", "sex": "M", "age": 43, "group": "hp"},
    {"initials": "TJ", "sex": "M", "age": 61, "group": "hp"},
    {"initials": "PW", "sex": "F", "age": 71, "group": "hp"},
    {"initials": "AR", "sex": "F", "age": 42, "group": "none"},
    {"initials": "JA", "sex": "F", "age": 58, "group": "none"},
    {"initials": "CA", "sex": "F", "age": 46, "group": "none"},
    {"initials": "SS", "sex": "M", "age": 55, "group": "none"},
    {"initials": "TL", "sex": "M", "age": 64, "group": "none"},
    {"initials": "LC", "sex": "F", "age": 39, "group": "none"},
    {"initials": "JE", "sex": "F", "age": 59, "group": "none"},
    {"initials": "MP", "sex": "M", "age": 64, "group": "none"},
    {"initials": "EA", "sex": "M", "age": 15, "group": "none"},
    {"initials": "MA", "sex": "F", "age": 13, "group": "none"},
    {"initials": "HA", "sex": "F", "age": 16, "group": "none"}
]

demographics_df = pd.DataFrame(demographics_data)

# Helper to extract clean initials from the participant ID to ensure we perfectly match
def extract_initials_for_merge(pid):
    if pd.isna(pid): 
        return None
    return str(pid).strip().split("-")[-1].upper()

# Ensure we're targeting the output dataframe that was successfully verified earlier
output_df_enrich = output_df.copy()
output_df_enrich['merge_initials'] = output_df_enrich['participant_id'].apply(extract_initials_for_merge)

# Perform Left Join to pull in age and sex
enriched_df = output_df_enrich.merge(
    demographics_df[['initials', 'sex', 'age']], 
    left_on='merge_initials', 
    right_on='initials', 
    how='left'
)

# Clean up merging keys
enriched_df.drop(columns=['merge_initials', 'initials'], inplace=True, errors='ignore')

# Reorder columns to group participant_id, age, sex together nicely 
cols = list(enriched_df.columns)
front_cols = ['filename', 'participant_id', 'DP_or_HP', 'age', 'sex']
ordered_cols = front_cols + [c for c in cols if c not in front_cols]
enriched_df = enriched_df[ordered_cols]

# Save the brand new Excel file!
OUTPUT_AGE_FILE = Path.cwd() / "excel_filer" / "matched_filenames_with_demographics.xlsx"
enriched_df.to_excel(OUTPUT_AGE_FILE, index=False)

print(f"Success! Demographic data added.")
print(f"Saved strictly to: {OUTPUT_AGE_FILE.name}\n")   
display(enriched_df[['filename', 'participant_id', 'age', 'sex', 'scan_number']].head(10))

Success! Demographic data added.
Saved strictly to: matched_filenames_with_demographics.xlsx



,filename,participant_id,age,sex,scan_number
0,AC_Right Hallux_L1051_S2392__20_05_2021.dcm,DP-07-AC,57.0,M,2392
1,AC_Right Hallux_L1051_S2629__24_05_2021.dcm,NaN,NaN,NaN,2629
2,AC_Right Hallux_L1051_S2432__20_05_2021.dcm,HP-24-AKD,26.0,M,2432
3,AC_Right Hallux_L1051_S2597__24_05_2021.dcm,DP-07-AC,57.0,M,2597
4,AT_Right Hallux_L1095_S3454__31_05_2021.dcm,DP-14-AT,58.0,M,3454
5,AC_Right Hallux_L1051_S2598__24_05_2021.dcm,DP-07-AC,57.0,M,2598
6,AC_Right Hallux_L1051_S2602__24_05_2021.dcm,DP-07-AC,57.0,M,2602
7,AC_Right Hallux_L1051_S2626__24_05_2021.dcm,NaN,NaN,NaN,2626
8,AC_Right Hallux_L1051_S2621__24_05_2021.dcm,NaN,NaN,NaN,2621
9,AC_Right Hallux_L1051_S2440__20_05_2021.dcm,HP-24-AKD,26.0,M,2440


# October

In [7]:
# =========================================================
# PROCESS OCTOBER (Diff Layout)
# =========================================================
OCT_EXCEL_FILE = Path.cwd() / "excel_filer" / "OCTOBER.xlsx"
OCT_DATA_PATH = Path(r"D:\OCTOBER") # Given parameter

df_oct1 = pd.read_excel(OCT_EXCEL_FILE, sheet_name="Sheet1")
df_oct2 = pd.read_excel(OCT_EXCEL_FILE, sheet_name="Sheet2")

# We want to create exactly the same format so the previous regex works
oct_long_rows = []

# --- PARSE SHEET 1 ---
# Ffill the participant ID since it acts as a header block
df_oct1["Perticipant ID"] = df_oct1["Perticipant ID"].ffill()

for idx, row in df_oct1.iterrows():
    participant_id = row.get("Perticipant ID")
    if pd.isna(participant_id): continue
    
    initials = str(participant_id).strip().upper()

    # Get Demographics data from our data dictionary if they exist there
    # (Checking the existing python dict list we made before)
    match_demos = [d for d in demographics_data if d['initials'] == initials]
    group = match_demos[0]['group'].upper() if match_demos else "Unknown"

    # Block 1 (Left Side)
    b1_setting = row.get("Imaging settings")
    b1_scan = row.get("Scan #")
    
    # Block 2 (Right Side)
    b2_setting = row.get("Imaging settings.1")
    b2_scan = row.get("Scan #.1")
    
    # Function to parse typical Oct settings: 'HB' -> Hand Baseline, 'FI' -> Foot Isc
    def parse_setting(setting_str):
        if pd.isna(setting_str): return None, None
        s = str(setting_str).strip().upper()
        if len(s) < 2: return None, None
        area = "Hand" if s[0] == "H" else "Foot" if s[0] == "F" else "Unknown"
        phase = "Baseline" if s[1] == "B" else "Isc" if s[1] == "I" else "PORH" if s[1] == "P" else "Unknown"
        return area, phase
        
    for current_setting, current_scan in [(b1_setting, b1_scan), (b2_setting, b2_scan)]:
        if pd.isna(current_scan) or str(current_scan).strip() == "_" or str(current_scan).strip() == "":
            continue
            
        area, phase = parse_setting(current_setting)
        
        # We need to expand ranges like '133-145' just in case October has them
        scans = expand_scan_field(str(current_scan))
        
        for s in scans:
            oct_long_rows.append({
                "participant_id": initials, # the october sheet doesn't prefix "HP-XX-"
                "DP_or_HP": group,
                "initials": initials,
                "date": pd.to_datetime("2020-10-01"), # Fallback generic date if empty
                "scan_number": int(s),
                "phase": phase,
                "protocol_area": area,
                "condition": "Regular",
                "excel_column": current_setting
            })

# --- PARSE SHEET 2 ---
# Sheet 2 is loose. "Additional Data" contains the Patient if length < 4, otherwise category.
df_oct2["Additional Data"] = df_oct2["Additional Data"].ffill()
df_oct2["Unnamed: 1"] = df_oct2["Unnamed: 1"].ffill() # Sometimes area / phase like Hand/Foot/Laying Down

for idx, row in df_oct2.iterrows():
    c1 = str(row.get("Additional Data", "")).strip()
    c2 = str(row.get("Unnamed: 1", "")).strip()
    scan_val = row.get("Unnamed: 2")
    
    # Skip header rows
    if c1 == "Seating Positions" or pd.isna(scan_val) or scan_val == "Scan #" or c1 == "nan":
        continue
        
    initials = c1.upper()
    
    # Check demographics
    match_demos = [d for d in demographics_data if d['initials'] == initials]
    group = match_demos[0]['group'].upper() if match_demos else "Unknown"
    
    # Derive area (Could be Hand, Foot, or just generic from "Laying Down")
    area = "Unknown"
    if "Hand" in c2: area = "Hand"
    elif "Foot" in c2: area = "Foot"
    
    scans = expand_scan_field(str(scan_val))
    
    for s in scans:
        oct_long_rows.append({
            "participant_id": initials, 
            "DP_or_HP": group,
            "initials": initials,
            "date": pd.to_datetime("2020-10-01"),
            "scan_number": int(s),
            "phase": c2, # Might contain 'Sitting up', 'Legs vertical' etc.
            "protocol_area": area,
            "condition": "Additional",
            "excel_column": "Sheet2_Custom"
        })

oct_excel_df = pd.DataFrame(oct_long_rows)
print(f"Successfully converted October Excel into {len(oct_excel_df)} long rows!")
display(oct_excel_df.head())

Successfully converted October Excel into 479 long rows!


,participant_id,DP_or_HP,initials,date,scan_number,phase,protocol_area,condition,excel_column
0,RM,HP,RM,2020-10-01,120,Baseline,Hand,Regular,HB
1,RM,HP,RM,2020-10-01,134,Baseline,Foot,Regular,FB
2,RM,HP,RM,2020-10-01,121,Baseline,Hand,Regular,HB
3,RM,HP,RM,2020-10-01,135,Baseline,Foot,Regular,FB
4,RM,HP,RM,2020-10-01,122,Baseline,Hand,Regular,HB


In [8]:
# =========================================================
# PARSE OCTOBER FILENAMES & MERGE
# =========================================================
oct_file_rows = []
oct_unmatched_patterns = []

# Assuming the same filename pattern works perfectly here too
for fname in tqdm.tqdm(os.listdir(OCT_DATA_PATH), desc="Parsing OCT DICOM files"):
    if fname.startswith("."): continue
    
    m = filename_pattern.search(fname.strip()) 
    
    if not m:
        fallback_m = backup_pattern.search(fname.strip())
        if fallback_m:
            oct_file_rows.append({
                "filename": fname,
                "initials": "UNKNOWN",  
                "side": "Unknown",
                "bodypart": "Unknown",
                "protocol_area": "Unknown",
                "Lnum": "Unknown",
                "scan_number": int(fallback_m.group("scan_number")),
                "date": pd.to_datetime(fallback_m.group("date"), format="%d_%m_%Y", errors="coerce").normalize(),
            })
        else:
            oct_unmatched_patterns.append(fname)
        continue
        
    bodypart = m.group("bodypart") if m.group("bodypart") else ""
    bodypart = bodypart.strip()
    
    # Handle dates
    date_str = m.group("date")
    if "_" in date_str:
        file_date = pd.to_datetime(date_str, format="%d_%m_%Y", errors="coerce")
    else:
        file_date = pd.to_datetime(date_str, errors="coerce") 
    
    # Map protocol area (Instep -> Foot)
    protocol_area = "Hand" if "hand" in bodypart.lower() or "finger" in bodypart.lower() else "Foot"
    
    initials = m.group("initials") if m.group("initials") else ""
    initials = initials.upper().replace("-", "")
    
    oct_file_rows.append({
        "filename": fname,
        "initials": initials,
        "side": m.group("side") if m.group("side") else "Unknown",
        "bodypart": bodypart,
        "protocol_area": protocol_area,
        "Lnum": m.group("Lnum") if m.group("Lnum") else "Unknown",
        "scan_number": int(m.group("scan_number")),
        "date": file_date.normalize() if pd.notna(file_date) else pd.NaT,
    })

oct_files_df = pd.DataFrame(oct_file_rows)

print(f"\nSuccessfully parsed {len(oct_files_df)} files.")

# Merge time!
# Just like before, we merge on initials and scan_number. Since October has missing protocol_areas occasionally we might just match on scan number directly if initials match
oct_matched_df = oct_files_df.merge(oct_excel_df, on=["initials", "scan_number"], how="left", suffixes=('_file', '_excel'))

# Identify files that still didn't match
unmatched_mask = oct_matched_df['participant_id'].isna()

if unmatched_mask.any():
    unmatched_subset = oct_matched_df[unmatched_mask].drop(columns=['participant_id','DP_or_HP','phase','condition','date_excel','excel_column', 'protocol_area_excel'], errors='ignore')
    
    # fuzzy matches 
    fuzzy_matches = unmatched_subset.merge(
        oct_excel_df, 
        on=["scan_number"], # just pure scan number fallback since October is small
        how="inner", 
        suffixes=('_file', '_excel')
    )
    fuzzy_matches = fuzzy_matches.drop_duplicates(subset=['filename'])
    
    for _, fuzzy_row in fuzzy_matches.iterrows():
        idx = oct_matched_df[oct_matched_df['filename'] == fuzzy_row['filename']].index
        if len(idx) > 0:
            oct_matched_df.loc[idx, 'participant_id'] = fuzzy_row['participant_id']
            oct_matched_df.loc[idx, 'DP_or_HP'] = fuzzy_row['DP_or_HP']
            oct_matched_df.loc[idx, 'phase'] = fuzzy_row['phase']
            oct_matched_df.loc[idx, 'condition'] = fuzzy_row['condition']
            oct_matched_df.loc[idx, 'excel_column'] = fuzzy_row['excel_column']
            oct_matched_df.loc[idx, 'protocol_area_excel'] = fuzzy_row['protocol_area_excel'] if 'protocol_area_excel' in fuzzy_row else fuzzy_row.get('protocol_area_y', fuzzy_row.get('protocol_area'))

# Enrich with Demographics immediately!
oct_matched_df['merge_initials'] = oct_matched_df['participant_id'].apply(lambda x: str(x).strip().upper() if pd.notna(x) else None)

oct_enriched_df = oct_matched_df.merge(
    demographics_df[['initials', 'sex', 'age']], 
    left_on='merge_initials', 
    right_on='initials', 
    how='left'
)
oct_enriched_df.drop(columns=['merge_initials', 'initials_y'], inplace=True, errors='ignore')
if 'initials_x' in oct_enriched_df.columns:
    oct_enriched_df.rename(columns={'initials_x': 'initials'}, inplace=True)

# Organize
cols = list(oct_enriched_df.columns)
front_cols = ['filename', 'participant_id', 'DP_or_HP', 'age', 'sex', 'phase', 'scan_number']
ordered_cols = front_cols + [c for c in cols if c not in front_cols]
oct_enriched_df = oct_enriched_df[ordered_cols]

# Save Match file
OCTOBER_MATCH_OUTPUT = Path.cwd() / "excel_filer" / "october_matched_filenames_with_demographics.xlsx"
oct_enriched_df.to_excel(OCTOBER_MATCH_OUTPUT, index=False)

# Save Unmatched
OCTOBER_UNMATCHED_OUTPUT = Path.cwd() / "excel_filer" / "october_unmatched.xlsx"
oct_unmatched_df = oct_enriched_df[oct_enriched_df["participant_id"].isna()].copy()
oct_unmatched_df.to_excel(OCTOBER_UNMATCHED_OUTPUT, index=False)

print(f"\n--- OCTOBER SUMMARY ---")
print(f"Total parsed items: {len(oct_files_df)}")
print(f"Successfully tracked: {oct_enriched_df['participant_id'].notna().sum()}")
print(f"Untracked: {len(oct_unmatched_df)}")
display(oct_enriched_df[['filename', 'participant_id', 'scan_number', 'age', 'sex', 'phase']].head(10))

Parsing OCT DICOM files: 100%|██████████| 556/556 [00:00<00:00, 6072.13it/s]



Successfully parsed 555 files.

--- OCTOBER SUMMARY ---
Total parsed items: 555
Successfully tracked: 478
Untracked: 78


,filename,participant_id,scan_number,age,sex,phase
0,AF_Right Hand_L1017_S288__07_10_2020.dcm,AF,288,33.0,M,PORH
1,AF_Right Hand_L1017_S289__07_10_2020.dcm,AF,289,33.0,M,PORH
2,AF_Right Instep_L1018_S290__07_10_2020.dcm,AF,290,33.0,M,Baseline
3,AF_Right Instep_L1018_S291__07_10_2020.dcm,AF,291,33.0,M,Baseline
4,AF_Right Instep_L1018_S292__07_10_2020.dcm,AF,292,33.0,M,Baseline
5,AF_Right Instep_L1018_S293__07_10_2020.dcm,AF,293,33.0,M,Baseline
6,AF_Right Instep_L1018_S294__07_10_2020.dcm,AF,294,33.0,M,Baseline
7,AF_Right Instep_L1018_S295__07_10_2020.dcm,AF,295,33.0,M,Isc
8,AF_Right Instep_L1018_S296__07_10_2020.dcm,AF,296,33.0,M,PORH
9,AF_Right Instep_L1018_S297__07_10_2020.dcm,AF,297,33.0,M,PORH
